# 37 Dashboard: Handlungsempfehlungen auf dem Rohrbaum

Teil 9: Dashboard

**Worum es geht**
- Das fertige Dashboard: Es zeigt auf dem Rohrbaum aus Notebook 32 für jede Hausstation, was die Modelle aus Notebook 35 sagen.
- Es deckt die vier Aufgaben ab:
  - Datenanalyse: Netzeffizienz, Stand jeder Station im Netz, Meldungen bei überhöhtem Rücklauf oder Dauerdurchfluss.
  - Modellentwicklung: Meldungen der Wächter (Anomalien) und Verdacht auf die Fehlerart Ventil, Dämmung oder zu großer Anschluss.
  - Visualisierung: Karte mit Zeitregler, gefärbt nach Dringlichkeit, Maßnahme, Meldungen, Potenzial oder Messwerten.
  - Empfehlungssystem: Handlungsliste mit Maßnahme, Dringlichkeit und Begründung je Zähler. Regelungsoptimierung empfiehlt das Dashboard bewusst nicht, weil kein Modell sie erkennt (Notebook 35, Test C).
- Welches Modell wofür (Details im Reiter „So funktioniert's“ und in Notebook 35, Abschnitt 20 „So rechnet das Dashboard“):
  - Veränderung an Heiztagen: Tagesmonitor (Notebook 20).
  - Veränderung an warmen Tagen: Prognose-LSTM (Notebook 28).
  - Ventil und Dämmung: LSTM-Fehlerklassen (Notebook 27).
  - Zu großer Anschluss: Regel Spitzenlast zum Vertrag.
  - Stationsleistung und Potenzial: Netzvergleich (Notebook 19).
- Damit es nicht überladen wirkt, zeigt die Karte immer nur eine Färbung und das Hausfenster immer dieselben Punkte (Empfehlung, Leistung im Netz, Verdacht, Veränderung, Befund). Erklärungen stehen im eigenen Reiter, Modellvergleiche und AUCs bleiben in Notebook 35.

**Daten**
- liest: `daten_cache/paket.pkl` aus Notebook 31 (braucht die Koordinatenliste aus `data/manual/`), Modellergebnisse in `outputs/dashboard/`
- schreibt: `fernwaerme_dashboard_final.html` in den Ordner `notebooks/`, Doppelklick öffnet sie im Browser. Die Datei enthält Adressen und wird nicht weitergegeben.
- vorher ausführen: Notebook 31 einmal. Für `outputs/dashboard/` entweder `python -m src.dashboard_daten` im Repo-Ordner (braucht die Modellpakete in `models/` aus Notebook 35, Abschnitt „Modelle speichern und Dashboard umstellen“) oder den Ordner von jemandem kopieren, der die Pakete hat. Er enthält nur Zählernummern, keine Adressen.
- ausliefern: Die fertige HTML-Datei lässt sich auch per Docker bereitstellen (`docker compose up dashboard`, Details in der README).

**Code**
- Rohrbaum wie in Notebook 32, Abschnitte 3 bis 5.
- Karte in `dashboard_final.py`, einer Kopie von `dashboard_export.py`. Das Original bleibt unverändert.
- Die Modellergebnisse liest `src/dashboard_karte.py`, gerechnet werden sie in `src/dashboard_daten.py`.

In [ ]:
import os as _os
from pathlib import Path as _Path
#Arbeitsordner ist immer notebooks/, auch aus den Teil-Ordnern
_os.chdir(next(p for p in [_Path.cwd(), *_Path.cwd().parents] if (p / "src").is_dir()) / "notebooks")
import sys
import runpy
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
from pyproj import Transformer
from scipy.spatial import cKDTree

## 1 Repo und Daten laden

In [ ]:
#Repo-Ordner suchen
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dashboard_karte.py").exists()), None)
if ROOT is None:
    raise SystemExit("Repo-Ordner nicht gefunden: Notebook aus dem Ordner notebooks/ starten oder ROOT von Hand setzen.")
sys.path.insert(0, str(ROOT))
print("Repo:", ROOT)

In [ ]:
#vorher Notebook 31 ausführen

ORDNER = None      #Pfad eintragen, falls die Suche nichts findet

def finde_cache():
    hier = Path.cwd().resolve()
    kandidaten = [hier / "daten_cache", hier.parent / "daten_cache", *sorted(hier.parent.glob("*/daten_cache"))]
    for k in kandidaten:
        if (k / "paket.pkl").exists():
            return k

ORDNER = Path(ORDNER) if ORDNER else finde_cache()
if ORDNER is None or not (ORDNER / "paket.pkl").exists():
    raise SystemExit("paket.pkl nicht gefunden. Notebook 31 ausführen und den Pfad aus der Zeile "
                     "'Gespeichert in ...' oben bei ORDNER eintragen.")
print("Lade aus:", ORDNER, flush=True)

paket = pd.read_pickle(ORDNER / "paket.pkl")
globals().update(paket)

for pfad in (ORDNER.parent, Path(".").resolve()):
    if str(pfad) not in sys.path:
        sys.path.insert(0, str(pfad))
norm_z = lambda s: "".join(ch for ch in str(s) if ch.isdigit()).lstrip("0")

print(f"Zähler mit Zeitreihe: {len(serien)} | Nodes im Graph: {G.number_of_nodes()} | Kanten: {G.number_of_edges()}")
fehlt = [n for n in ("serien", "knoten", "G", "pos", "roh", "linien", "kml_linien") if n not in globals()]
print("Für den Dashboard-Export fehlt:", fehlt or "nichts")

### 1.1 Modellergebnisse

- Die Modelle rechnet `python -m src.dashboard_daten` mit den Paketen aus Notebook 35. Hier wird nur geprüft, ob die Ergebnisse da sind und von wann sie stammen.
- Sind neue Messdaten dazugekommen, `NEU_RECHNEN = True` setzen. Das dauert etwa eine Minute und braucht die Pakete in `models/`.

In [ ]:
from src import dashboard_karte as dk

NEU_RECHNEN = False        #True: Modellpakete auf die aktuellen Daten anwenden
MODELL_ORDNER = ROOT / "outputs" / "dashboard"

if NEU_RECHNEN or not (MODELL_ORDNER / "stationen.parquet").exists():
    from src import dashboard_daten, modellpaket
    try:
        dashboard_daten.rechne(MODELL_ORDNER)
    except modellpaket.PaketFehler as ex:
        raise SystemExit(f"{ex}\nDie Pakete liegen nicht im Repo (models/ ist gitignored). Entweder Notebook 35, "
                         "Abschnitt 17 ausführen oder den Ordner outputs/dashboard/ von jemandem kopieren, der sie hat.")

modell = dk.lade(MODELL_ORDNER)
meta = modell["meta"]
print("Modellpakete:", meta["pakete"])
print("Gerechnet:", meta["erstellt"], "| Messdaten bis:", str(meta["datenende"])[:10], "| Winter:", meta["winter_name"])
print("Zähler mit Ergebnis:", meta["stationen"], "| Dringlichkeit:", meta["dringlichkeit"])

## 2 Rohrbaum vom Erzeuger (aus Notebook 32)

- Gleiche Schritte wie Notebook 32, Abschnitte 3 bis 5. Die Prüfungen (stimmt die Fließrichtung, ist das obere Haus wärmer) und die Bilder stehen dort.
- Der Dashboard-Export aus Notebook 31 läuft einmal mit, weil er das Rohrnetz `Gp` und die Zuordnung der Häuser zum Rohr (`snap`) berechnet. Er schreibt dabei auch das alte `fernwaerme_dashboard.html`.

In [ ]:
%run -i dashboard_export.py

In [ ]:
#Erzeuger als Quelle, Leitungen von ihm weg ausrichten (NB 32, Abschnitt 3)
QUELLE = "Erzeuger (Annahme)"   #von Experten bestätigt: dieser Knoten ist die Einspeisung
ADRESSE = "hans-zander-weg 14"  #Node mit der echten Adresse des Erzeugers (Spalte "node")

if QUELLE not in G:
    raise SystemExit(f"'{QUELLE}' fehlt im Netz: Zelle mit dashboard_export.py erneut ausführen.")

if ADRESSE in G:
    roh[QUELLE] = f"Erzeuger ({roh.get(ADRESSE, ADRESSE)})"
    G = nx.contracted_nodes(G, QUELLE, ADRESSE, self_loops=False, copy=True)
    pos.pop(ADRESSE, None)

U = G.to_undirected()
tiefe = nx.single_source_shortest_path_length(U, QUELLE)
G_neu = nx.DiGraph()
G_neu.add_nodes_from(G.nodes(data=True))
for u, v, d in G.edges(data=True):
    if u in tiefe and v in tiefe:
        a, b = (u, v) if (tiefe[u], str(u)) <= (tiefe[v], str(v)) else (v, u)
    else:
        a, b = u, v
    G_neu.add_edge(a, b, **d)
G = G_neu
print(f"Nodes vom Erzeuger nicht erreichbar: {sum(1 for n in G if n not in tiefe)} | "
      f"Nodes ohne eingehende Kante: {sum(1 for n in G if G.in_degree(n) == 0)} (im Idealfall nur der Erzeuger)")

In [ ]:
#Zähler je Node, Erzeuger auf den nächsten Rohrpunkt (NB 32, Abschnitt 4)
nz = norm_z if "norm_z" in globals() else (lambda s: "".join(ch for ch in str(s) if ch.isdigit()).lstrip("0"))
zaehler_von = {r.edge_node: [nz(z) for z in str(r.zaehler).split(",") if nz(z)] for r in knoten.itertuples()}

t = Transformer.from_crs(4326, 25832, always_xy=True)
qx, qy = t.transform(pos[QUELLE][1], pos[QUELLE][0])
d_q, i_q = cKDTree(np.array(V)).query([qx, qy])
q_v = V[i_q]
print(f"Erzeuger liegt {d_q:.0f} m neben dem nächsten Rohrpunkt")

In [ ]:
#Rohrbaum: kürzester Rohrweg vom Erzeuger zu jedem Haus (NB 32, Abschnitt 5)
haeuser = [n for n in G if n != QUELLE and n in nid and zaehler_von.get(n)]
dist_q, pfade = nx.single_source_dijkstra(Gp, q_v, weight="len")

snap_von = {h: snap[nid[h]] for h in haeuser}
ohne_weg = [h for h, s in snap_von.items() if s not in pfade]
snap_von = {h: s for h, s in snap_von.items() if s in pfade}

#1) Baum: Wege vom Erzeuger zu allen Anschlussstellen
parent = {}
for s in set(snap_von.values()):
    p = pfade[s]
    for a, b in zip(p[:-1], p[1:]):
        parent[b] = a
kinder = {}
for b, a in parent.items():
    kinder.setdefault(a, []).append(b)
haus_an = {}
for h, s in snap_von.items():
    haus_an.setdefault(s, []).append(h)

#2) Schlüsselpunkte: Erzeuger, Anschlussstellen, Verzweigungen
schluessel = {q_v} | set(haus_an) | {v for v, k in kinder.items() if len(k) >= 2}

#3) Straßenabschnitte zwischen Schlüsselpunkten
T = nx.DiGraph()
for v in schluessel:
    T.add_node(v, art="quelle" if v == q_v else ("haus_stelle" if v in haus_an else "abzweig"))
for v in schluessel - {q_v}:
    weg = [v]
    u = parent[v]
    while u not in schluessel:
        weg.append(u)
        u = parent[u]
    weg.append(u)
    T.add_edge(u, v, laenge=dist_q[v] - dist_q[u], pfad=weg[::-1])

print(f"Häuser mit Zähler: {len(haeuser)} | ohne Rohrweg zum Erzeuger: {len(ohne_weg)}")
print(f"Rohrbaum: {T.number_of_nodes()} Schlüsselpunkte | {T.number_of_edges()} Abschnitte | "
      f"{sum(d['laenge'] for *_, d in T.edges(data=True)):.0f} m Rohr genutzt")

## 3 Dashboard bauen

- Graph wie in Notebook 32: Erzeuger, Einspeisepunkt am Rohr, Abzweige und Anschlussstellen, Häuser.
- `dashboard_final.py` läuft in einem eigenen Namensraum, damit die Variablen hier (`nid`, `snap`, `Gp`) nicht überschrieben werden.
- In der zweiten Zelle stehen der Zeitraum der Karte und die Wahl, ob Abzweige als graue Punkte erscheinen.

In [ ]:
t_ll = Transformer.from_crs(25832, 4326, always_xy=True)
def ll_punkt(p):
    lo, la = t_ll.transform(*p)
    return (la, lo)

name_von = {v: f"Rohr {i}" for i, v in enumerate(T.nodes)}
if set(name_von.values()) & set(G.nodes):
    raise SystemExit("Namenskonflikt: ein Node im Netz heißt schon 'Rohr …'.")
ART = {"quelle": "Einspeisepunkt am Rohr", "abzweig": "Abzweig", "haus_stelle": "Anschlussstelle"}

G_dash = nx.DiGraph()
pos_dash, roh_dash = dict(pos), dict(roh)
for v, d in T.nodes(data=True):
    G_dash.add_node(name_von[v])
    pos_dash[name_von[v]] = ll_punkt(v)
    roh_dash[name_von[v]] = ART[d["art"]]
G_dash.add_edges_from((name_von[u], name_von[v]) for u, v in T.edges)
G_dash.add_edge(QUELLE, name_von[q_v])
for s, hs in haus_an.items():
    for h in hs:
        if h in pos:
            G_dash.add_edge(name_von[s], h)

print(f"Dashboard-Graph: {G_dash.number_of_nodes()} Nodes, {G_dash.number_of_edges()} Kanten | "
      f"Baum: {nx.is_arborescence(G_dash)} | Quellen: {sum(1 for n in G_dash if G_dash.in_degree(n) == 0)}")

In [ ]:
START = "2024-01-01 00:00"     #Zeitraum der Karte
ENDE = _os.environ.get("DASHBOARD_ENDE", "2025-09-30 23:00")   #bis Ende Sommer 2025, damit die Meldungen des LSTM an warmen Tagen drin sind; python -m src.aktualisieren rückt das Ende mit den Daten nach
PUNKTE_ZEIGEN = False          #Abzweige und Anschlussstellen als graue Punkte zeigen?
AUSGABE = "fernwaerme_dashboard_final.html"

ns = runpy.run_path(str(ROOT / "notebooks" / "dashboard_final.py"), init_globals={
    "serien": serien, "knoten": knoten, "G": G_dash, "pos": pos_dash, "roh": roh_dash,
    "ROOT": ROOT, "MODELL_ORDNER": MODELL_ORDNER, "START": START, "ENDE": ENDE,
    "PUNKTE_ZEIGEN": PUNKTE_ZEIGEN, "AUSGABE": AUSGABE,
    **{k: globals()[k] for k in ("norm_z", "linien", "kml_linien", "geschaetzt") if k in globals()}})

## 4 Kontrolle

- Zähler mit Modellergebnis, die nicht auf der Karte stehen, erscheinen trotzdem in der Handlungsliste, mit dem Vermerk „nicht auf der Karte“.
- Häuser auf der Karte ohne Modellergebnis haben zu wenige Daten im letzten Winter.
- Die Tabelle zeigt die ersten zehn Zeilen der Handlungsliste, so wie sie im Dashboard stehen.

In [ ]:
abgleich = ns["abgleich"]
print("Zähler mit Modellergebnis, aber nicht auf der Karte:", len(abgleich["modell_nicht_auf_karte"]))
print("Zähler auf der Karte ohne Modellergebnis:", len(abgleich["auf_karte_ohne_modell"]))

st = modell["stationen"].assign(zaehler=lambda d: d["zaehler"].map(dk.zaehler_text))
st["auf_karte"] = ~st["zaehler"].isin(abgleich["modell_nicht_auf_karte"])
pd.crosstab(st["dringlichkeit"], st["auf_karte"]).reindex(["hoch", "mittel", "niedrig", "keine"])

In [ ]:
st.sort_values("rang")[["rang", "zaehler", "dringlichkeit", "handlung", "begruendung"]].head(10)

## Zusammenfassung & Bewertung

**Ergebnis**
- Ein Dashboard mit drei Reitern auf dem Rohrbaum aus Notebook 32: Karte, Handlungsliste und „So funktioniert's“. Jede Anzeige kommt von dem Modell, das in Notebook 35 dafür am besten abgeschnitten hat.
- Fehlerarten und Maßnahmen: Ventil führt zu Reparatur, Dämmung zu einem Hinweis, zu großer Anschluss zu einer Kapazitätsanpassung. Regelung wird nicht empfohlen, weil kein Modell sie erkennt.
- Die Handlungsliste ist nach Dringlichkeit und dann nach Ventil-Score sortiert. Meldungen, Potenzial und eine Einspar-Schätzung ändern die Reihenfolge nicht, weil sie Befunde nicht vorhersagen.

**Belastbar**
- Die Reihenfolge nach Ventil-Score ist der Teil, den Notebook 35 belegt (Test E: 8 der ersten 10 mit Ventilbefund).

**Schwach**
- Dämmung erscheint oft. Sie ist bei 41 % der begangenen Stationen eingetragen, das Modell gibt dazu nur einen Hinweis.
- Empfehlung und Verdacht gelten für den letzten Winter und ändern sich nicht mit dem Zeitregler.
- Die Dringlichkeitsstufen sind gesetzt, nicht gelernt. Ob die Reihenfolge im Betrieb trägt, zeigt erst eine Begehung nach der Liste.
- Keine Meldung heißt nicht „in Ordnung“.

**Offen**
- Die Referenz des Tagesmonitors nach jeder Begehung neu lernen (Notebook 35, Abschnitt „Grenzen“). Danach `python -m src.dashboard_daten` und dieses Notebook neu ausführen.